# `Neo4jGraphRAGRetriever`: GraphRAG on an existing Neo4j graph with Haystack

This notebook uses the public, read-only Neo4j demo database (`companies`: news articles, chunks and organizations) to show:

1. The four search types: `vector`, `vector_cypher`, `hybrid`, `hybrid_cypher`
2. `query_params` in a `retrieval_query`
3. A RAG `Pipeline`
4. Saving and loading the pipeline as YAML
5. An `Agent` using the retriever as tools

The `haystack-neo4j[graphrag]` install adds only `neo4j-graphrag` to `haystack-ai` and `neo4j`. The embedder is your choice. Here we install `sentence-transformers-haystack`, because `all-MiniLM-L6-v2` matches the demo DB's `news_sbert` index.

In [ ]:
# From a clone of the repo (or: pip install "haystack-neo4j[graphrag] @ git+https://github.com/neo4j-labs/neo4j-haystack")
%pip install -q -e "..[graphrag]" sentence-transformers-haystack python-dotenv

## Credentials

The demo DB credentials are public. The OpenAI key is read from `.env` (or asked for with `getpass`) and never printed.

In [ ]:
import os
from getpass import getpass

from dotenv import find_dotenv, load_dotenv

os.environ.setdefault("NEO4J_USERNAME", "companies")
os.environ.setdefault("NEO4J_PASSWORD", "companies")

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

## 1. Four search types, one query

In [ ]:
from haystack_integrations.components.embedders.sentence_transformers import SentenceTransformersTextEmbedder

from haystack_integrations.components.retrievers.neo4j import Neo4jGraphRAGRetriever

URL = "neo4j+s://demo.neo4jlabs.com"

RETRIEVAL_QUERY = """
WITH node AS chunk, score
MATCH (article:Article)-[:HAS_CHUNK]->(chunk)
OPTIONAL MATCH (article)-[:MENTIONS]->(org:Organization)
OPTIONAL MATCH (org)-[:HAS_COMPETITOR]->(rival:Organization)
RETURN chunk.text AS text, article.id AS article_id, article.title AS title,
       toString(article.date) AS date, article.sentiment AS sentiment,
       collect(DISTINCT org.name)[..5] AS companies,
       collect(DISTINCT rival.name)[..5] AS competitors, score
ORDER BY score DESC
"""


def make_retriever(search_type, **kwargs):
    params = dict(
        url=URL,
        database="companies",
        search_type=search_type,
        vector_index_name="news_sbert",
        text_embedder=SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2"),
        cypher_version="25",  # the demo DB defaults to Cypher 5; neo4j-graphrag emits Cypher 25
        top_k=3,
    )
    if search_type.startswith("hybrid"):
        params["fulltext_index_name"] = "news_fulltext"
    if search_type.endswith("_cypher"):
        params["retrieval_query"] = RETRIEVAL_QUERY
    params.update(kwargs)
    return Neo4jGraphRAGRetriever(**params)


retrievers = {st: make_retriever(st) for st in ["vector", "vector_cypher", "hybrid", "hybrid_cypher"]}

In [ ]:
QUERY = "Which companies are working on electric vehicle batteries?"

for search_type, retriever in retrievers.items():
    docs = retriever.run(query=QUERY)["documents"]
    print(f"\n=== {search_type} ===")
    for doc in docs:
        print(f"score={doc.score:.3f} | {doc.content[:90]!r}")
        print("   meta:", doc.meta)

`vector` and `hybrid` return the chunk's own properties (embeddings stripped) plus `labels` and `element_id`. The `*_cypher` types return whatever the `retrieval_query` returns: article title, date, mentioned companies and their competitors.

## 2. `query_params`: filter by date

`retrieval_query` is fixed configuration. Values used in it (`$since`) are passed per call with `query_params`.

In [ ]:
SINCE_QUERY = """
WITH node AS chunk, score
MATCH (article:Article)-[:HAS_CHUNK]->(chunk)
WHERE article.date >= datetime($since)
RETURN chunk.text AS text, article.id AS article_id, article.title AS title,
       toString(article.date) AS date, score
ORDER BY score DESC
"""

recent = make_retriever("vector_cypher", retrieval_query=SINCE_QUERY, top_k=5)
for doc in recent.run(query=QUERY, query_params={"since": "2023-01-01"})["documents"]:
    print(doc.meta["date"][:10], "|", doc.meta["title"])
recent.close()

## 3. RAG pipeline

retriever → `ChatPromptBuilder` → `OpenAIChatGenerator`. The prompt shows graph context from `meta` when it's there.

In [ ]:
from haystack import Pipeline
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.dataclasses import ChatMessage

template = [
    ChatMessage.from_system(
        "Answer using only the context. Cite articles as [article_id] when an article_id is given."
    ),
    ChatMessage.from_user(
        """Context:
{% for doc in documents %}
---
{% if doc.meta.article_id %}[{{ doc.meta.article_id }}] {% endif %}{% if doc.meta.title %}{{ doc.meta.title }}{% endif %}{% if doc.meta.date %} ({{ doc.meta.date[:10] }}){% endif %}
{% if doc.meta.companies %}Companies: {{ doc.meta.companies | join(', ') }}
{% endif %}{% if doc.meta.competitors %}Competitors: {{ doc.meta.competitors | join(', ') }}
{% endif %}{{ doc.content }}
{% endfor %}

Question: {{ question }}"""
    ),
]

rag = Pipeline()
rag.add_component("retriever", make_retriever("hybrid_cypher", top_k=5))
rag.add_component("prompt_builder", ChatPromptBuilder(template=template, required_variables=["question", "documents"]))
rag.add_component("llm", OpenAIChatGenerator(model="gpt-4o-mini"))
rag.connect("retriever.documents", "prompt_builder.documents")
rag.connect("prompt_builder.prompt", "llm.messages")

question = "Which companies are working on EV batteries, and who are their competitors?"
result = rag.run({"retriever": {"query": question}, "prompt_builder": {"question": question}})
print(result["llm"]["replies"][0].text)

## 4. Agent with two retriever tools

`parameters=` exposes only `query` to the LLM. `retrieval_query` is never a tool argument, so the model can't run arbitrary Cypher.

In [ ]:
from haystack.components.agents import Agent
from haystack.tools import ComponentTool

QUERY_ONLY = {
    "type": "object",
    "properties": {"query": {"type": "string", "description": "What to search for"}},
    "required": ["query"],
}

graph_tool = ComponentTool(
    component=retrievers["hybrid_cypher"],
    name="company_graph_search",
    description=(
        "Search news chunks with graph context: article id, title, date, sentiment, mentioned companies "
        "and their competitors. Use for specific, analytical questions about companies; cite article ids."
    ),
    parameters=QUERY_ONLY,
)
theme_tool = ComponentTool(
    component=retrievers["vector"],
    name="news_theme_search",
    description="Semantic search over news text. Use for broad themes and trends.",
    parameters=QUERY_ONLY,
)

agent = Agent(
    chat_generator=OpenAIChatGenerator(model="gpt-4o-mini"),
    tools=[graph_tool, theme_tool],
    system_prompt="You answer questions about companies in the news. Use the tools, and cite article ids when you have them.",
)
agent.warm_up()


def ask(q):
    messages = agent.run(messages=[ChatMessage.from_user(q)])["messages"]
    used = [tc.tool_name for m in messages for tc in (m.tool_calls or [])]
    print(f"Q: {q}\nTools used: {used}\nA: {messages[-1].text}\n")


ask("Who are the competitors of the companies mentioned in news about EV batteries? Cite articles.")
ask("What broad themes show up in the news about artificial intelligence?")

## 5. Clean up

In [ ]:
for r in retrievers.values():
    r.close()
rag.get_component("retriever").close()